# Evalbids Order: ¿Funciona el nuevo orden de relevancia?
### Análisis del A/B Test · Workana Growth Team

---

> **TL;DR** — El nuevo algoritmo muestra mejoras consistentes en todo el funnel de conversión
> (Accepted Bid Rate +2,8 pp, Paid Rate +3,1 pp, calidad Gold+ +6,7 pp), pero el test corrió
> solo 18 días con **35% de poder estadístico**, insuficiente para declarar significancia.
> **Recomendación: extender ~6 semanas** para resolver con rigor si el efecto es real.

---

**Contexto:** Workana reordenó los freelancers en la página Evalbids usando un score ponderado
(gamificación 30%, experiencia en categoría 20%, subcategoría 20%, skills 10%, ranking 10%, otros 10%)
en lugar del orden legacy. La hipótesis: un orden por relevancia real facilita al cliente la
selección del freelancer, impactando en conversión.

| | Control | Test |
|---|---|---|
| **Descripción** | Orden legacy | Nuevo orden ponderado + filtros de calidad |
| **Proyectos** | 1.139 | 1.095 |
| **Período** | 3 – 21 julio 2025 (18 días) |
| **Categorías** | 10 subcategorías (Web Design, WordPress, SEO, E-commerce, Apps, etc.) |

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Paleta consistente: morado = Test, gris = Control
C_TEST, C_CTRL = '#6C5CE7', '#A0A0A0'
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (13, 5)})

# ── Carga de datos ────────────────────────────────────────────────────
FILE = r'input/ChallengeDataAnalystEvalbidsOrderDatos.xlsx'
abtests       = pd.read_excel(FILE, sheet_name='abtests')
projects      = pd.read_excel(FILE, sheet_name='projects')
bids          = pd.read_excel(FILE, sheet_name='bids')
threads       = pd.read_excel(FILE, sheet_name='threads')
accepted_bids = pd.read_excel(FILE, sheet_name='accepted_bids')
payments      = pd.read_excel(FILE, sheet_name='payments')
skills        = pd.read_excel(FILE, sheet_name='skills')  # catálogo de referencia

print(f'Datos cargados — {len(projects):,} proyectos, {len(bids):,} bids, '
      f'{len(accepted_bids):,} accepted bids, {len(payments):,} pagos')

In [ ]:
# ── Tabla maestra (un registro por proyecto) ─────────────────────────
master = projects.merge(
    abtests[['project_id', 'segment']], left_on='id', right_on='project_id', how='inner'
).drop(columns=['project_id'])
master['group'] = master['segment'].map({'default': 'Control', 'evalbidsNewOrder': 'Test'})
# Nota: el1 (EL1 = Engagement Level 1: el cliente respondió al menos un mensaje)
# viene como columna booleana nativa de la tabla projects

# Bids por proyecto
master = master.merge(
    bids.groupby('project_id')['id'].count().rename('n_bids').reset_index(),
    left_on='id', right_on='project_id', how='left'
).drop(columns=['project_id'], errors='ignore')
master['n_bids'] = master['n_bids'].fillna(0).astype(int)

# Accepted bid (sí/no)
master['has_accepted_bid'] = master['id'].isin(accepted_bids['project_id'].unique()).astype(int)

# Mensajes totales por proyecto
msg = threads.groupby('project_id').agg(total_messages=('total_messages', 'sum')).reset_index()
master = master.merge(msg, left_on='id', right_on='project_id', how='left').drop(columns=['project_id'], errors='ignore')
master['total_messages'] = master['total_messages'].fillna(0).astype(int)

# Pagos completados (via accepted_bids → payments)
ab_pay = accepted_bids[['id', 'project_id']].merge(
    payments[payments['status'] == 'paid'][['accepted_bid_id', 'gross_gmv']],
    left_on='id', right_on='accepted_bid_id', how='inner'
)
gmv = ab_pay.groupby('project_id').agg(paid_gmv=('gross_gmv', 'sum')).reset_index()
master = master.merge(gmv, left_on='id', right_on='project_id', how='left').drop(columns=['project_id'], errors='ignore')
master['paid_gmv'] = master['paid_gmv'].fillna(0)
master['has_paid'] = (master['paid_gmv'] > 0).astype(int)

# Estado productivo y semana calendario
master['is_productive'] = master['status'].isin(['working','escrowing','finished','rating']).astype(int)
master['created'] = pd.to_datetime(master['created'])
master['week'] = master['created'].dt.isocalendar().week.astype(int)

ctrl = master[master['group'] == 'Control']
test = master[master['group'] == 'Test']

print(f'Tabla maestra: {len(master):,} proyectos (Control {len(ctrl):,} · Test {len(test):,})')
print(f'Validaciones — Sin duplicados: {master.id.nunique() == len(master)} · '
      f'Coherencia paid⊂accepted: {(master[master.has_paid==1].has_accepted_bid==1).all()}')

## 1. Métrica de éxito y por qué la elegimos

**Métrica principal: Accepted Bid Rate** — proporción de proyectos donde el cliente acepta al menos una propuesta.

¿Por qué esta y no otra? Porque es la **conversión más directa a lo que cambiamos**. El experimento
modifica el orden en que el cliente ve los freelancers. Si ese orden es mejor, el cliente debería
encontrar más fácilmente al freelancer adecuado y aceptar su propuesta. No depende de métodos de pago
(como Paid Rate) ni de si el freelancer responde bien (como EL1). Es *la* acción que refleja si el
nuevo orden hizo su trabajo.

**Métricas complementarias:**

| Métrica | Qué nos dice |
|---|---|
| **Tasa EL1** (Engagement Level 1: el cliente respondió al menos un mensaje) | ¿El cliente siquiera interactúa? (engagement inicial, paso previo a la conversión) |
| **Paid Rate** | ¿La aceptación se convirtió en dinero real? (conversión monetaria) |
| **Tasa Productiva** | ¿El proyecto avanzó a trabajo real? (working/escrowing/finished/rating) |
| **GMV Promedio** | ¿Cambia el ticket promedio? (calidad del match en términos económicos) |
| **Mensajes Promedio** | ¿Hay más o menos interacción? (engagement vs. fricción) |
| **Bids Promedio** *(guardrail)* | No debería cambiar — los bids se envían *antes* de que el cliente vea el orden. Si cambia, algo está mal en la asignación. |

In [ ]:
# ── Funciones de testing estadístico ──────────────────────────────────
def z_test_prop(x_c, n_c, x_t, n_t):
    """Z-test para dos proporciones independientes."""
    p_c, p_t = x_c / n_c, x_t / n_t
    p_pool = (x_c + x_t) / (n_c + n_t)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
    z = (p_t - p_c) / se if se > 0 else 0
    pval = 2 * (1 - norm.cdf(abs(z)))
    se_d = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)
    diff = p_t - p_c
    return z, pval, diff, diff - 1.96*se_d, diff + 1.96*se_d

def welch_t(arr_c, arr_t):
    """T-test de Welch para dos muestras independientes."""
    t, pval = stats.ttest_ind(arr_t, arr_c, equal_var=False)
    d = arr_t.mean() - arr_c.mean()
    se = np.sqrt(arr_t.var(ddof=1)/len(arr_t) + arr_c.var(ddof=1)/len(arr_c))
    return t, pval, d, d - 1.96*se, d + 1.96*se

## 2. Resultados: ¿qué pasó?

Empecemos con la vista completa. Después profundizamos en lo que importa.

In [ ]:
# ── Tabla de resultados con tests estadísticos ────────────────────────
n_c, n_t = len(ctrl), len(test)
rows = []

for name, col in [('Accepted Bid Rate ★', 'has_accepted_bid'), ('Tasa EL1', 'el1'),
                   ('Paid Rate', 'has_paid'), ('Tasa Productiva', 'is_productive')]:
    z, p, d, lo, hi = z_test_prop(ctrl[col].sum(), n_c, test[col].sum(), n_t)
    c_r, t_r = ctrl[col].mean(), test[col].mean()
    rows.append([name, f'{c_r:.1%}', f'{t_r:.1%}', f'{d*100:+.2f} pp',
                 f'{(t_r/c_r-1)*100:+.1f}%', f'{p:.4f}',
                 f'[{lo*100:+.2f}, {hi*100:+.2f}]', '✓' if p < 0.05 else ''])

cp, tp = ctrl[ctrl.has_paid==1]['paid_gmv'], test[test.has_paid==1]['paid_gmv']
t_s, p, d, lo, hi = welch_t(cp, tp)
rows.append(['GMV Promedio (pagados)', f'${cp.mean():.0f}', f'${tp.mean():.0f}',
             f'{d:+.1f} USD', f'{(tp.mean()/cp.mean()-1)*100:+.1f}%', f'{p:.4f}',
             f'[{lo:+.0f}, {hi:+.0f}] USD', '✓' if p < 0.05 else ''])

for name, col in [('Mensajes Promedio', 'total_messages'), ('Bids Promedio (guardrail)', 'n_bids')]:
    t_s, p, d, lo, hi = welch_t(ctrl[col], test[col])
    rows.append([name, f'{ctrl[col].mean():.1f}', f'{test[col].mean():.1f}',
                 f'{d:+.2f}', f'{(test[col].mean()/ctrl[col].mean()-1)*100:+.1f}%',
                 f'{p:.4f}', f'[{lo:+.2f}, {hi:+.2f}]', '✓' if p < 0.05 else ''])

results = pd.DataFrame(rows, columns=['Métrica','Control','Test','Δ','Lift','p-value','IC 95%','Sig.'])
print('─' * 100)
print('  RESULTADOS   │   z-test (proporciones)  ·  Welch t-test (medias)  ·  α = 0,05')
print('─' * 100)
print(results.to_string(index=False))
print('─' * 100)
print('★ Métrica principal')

In [ ]:
# ── Visualización: tasas de conversión y métricas de volumen ──────────
rate_data = [
    ('Accepted Bid Rate\n(★ principal)', ctrl.has_accepted_bid.mean()*100, test.has_accepted_bid.mean()*100),
    ('Tasa EL1', ctrl.el1.mean()*100, test.el1.mean()*100),
    ('Paid Rate', ctrl.has_paid.mean()*100, test.has_paid.mean()*100),
    ('Tasa Productiva', ctrl.is_productive.mean()*100, test.is_productive.mean()*100),
]

fig, axes = plt.subplots(1, 4, figsize=(17, 5.5))
for i, (title, vc, vt) in enumerate(rate_data):
    ax = axes[i]
    bars = ax.bar(['Control', 'Test'], [vc, vt], color=[C_CTRL, C_TEST],
                  width=0.5, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, [vc, vt]):
        ax.text(b.get_x()+b.get_width()/2, v+0.4, f'{v:.1f}%',
                ha='center', va='bottom', fontweight='bold', fontsize=12)
    lift = (vt/vc - 1) * 100
    color = '#2E7D32' if lift > 0 else '#C62828'
    ax.text(0.5, 0.94, f'Lift: {lift:+.1f}%', transform=ax.transAxes, ha='center',
            fontsize=10, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f8f7ff', ec='#ddd', alpha=0.9))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vc, vt) * 1.28); ax.set_ylabel('')

plt.suptitle('Métricas de Conversión: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/01_rates_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Métricas de volumen ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
vol_data = [
    ('Bids Promedio', ctrl.n_bids.mean(), test.n_bids.mean(), ''),
    ('Mensajes Promedio', ctrl.total_messages.mean(), test.total_messages.mean(), ''),
    ('GMV Promedio (USD)', cp.mean(), tp.mean(), '$'),
]
for i, (title, vc, vt, prefix) in enumerate(vol_data):
    ax = axes[i]
    bars = ax.bar(['Control','Test'], [vc, vt], color=[C_CTRL, C_TEST],
                  width=0.5, edgecolor='white', linewidth=1.5)
    for b, v in zip(bars, [vc, vt]):
        ax.text(b.get_x()+b.get_width()/2, v+max(vc,vt)*0.02,
                f'{prefix}{v:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    lift = (vt/vc - 1) * 100
    clr = '#2E7D32' if lift > 0 else '#C62828'
    ax.text(0.5, 0.94, f'Lift: {lift:+.1f}%', transform=ax.transAxes, ha='center',
            fontsize=10, color=clr, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='#f8f7ff', ec='#ddd', alpha=0.9))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vc, vt) * 1.25)

plt.suptitle('Métricas de Volumen: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/02_volume_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

**Primera lectura:** Todas las métricas de conversión suben en Test (lifts de +6% a +17%).
Sin embargo, ninguna cruza el umbral de significancia (α = 0,05). Las más cercanas:
EL1 (p = 0,056) y Paid Rate (p = 0,066). El guardrail (Bids Promedio) permanece estable (+2%, p = 0,59),
lo que confirma que la asignación aleatoria funcionó correctamente.

La única métrica significativa es Mensajes Promedio (p = 0,039): Test genera +6,5 mensajes por proyecto.
Combinado con la mejora en EL1 y conversión, esto apunta a **más engagement genuino**, no a fricción.

*Nota sobre comparaciones múltiples:* con 7 tests simultáneos, aplicando corrección de Bonferroni
(α/7 = 0,007) ninguna métrica alcanzaría significancia. Esto refuerza la necesidad de más muestra,
no de descartar la señal — las métricas están correlacionadas (forman un funnel) y la consistencia
direccional tiene valor interpretativo más allá de los p-values individuales.

¿Es ruido o señal? Profundicemos.

## 3. El funnel completo: ¿dónde está la mejora?

Si el nuevo orden funciona, deberíamos ver mejoras no solo al final del funnel,
sino *en cada etapa de paso*.

In [ ]:
# ── Funnel: tasas absolutas y de paso ─────────────────────────────────
def funnel(df):
    return [len(df), df.el1.sum(), df.has_accepted_bid.sum(), df.has_paid.sum()]

cf, tf = funnel(ctrl), funnel(test)
stages = ['Proyecto', 'EL1', 'Accepted Bid', 'Pago']

print('FUNNEL: tasas absolutas y tasas de paso entre etapas')
print('─' * 88)
print(f'{"Etapa":>15}  {"Control (n)":>12} {"(% abs)":>8} {"(% paso)":>9}  '
      f'{"Test (n)":>10} {"(% abs)":>8} {"(% paso)":>9}  {"Δ paso":>7}')
print('─' * 88)
for i, s in enumerate(stages):
    ca = cf[i]/cf[0]*100
    ta = tf[i]/tf[0]*100
    cs = cf[i]/cf[i-1]*100 if i > 0 else 100.0
    ts = tf[i]/tf[i-1]*100 if i > 0 else 100.0
    delta = f'{ts-cs:+.1f}pp' if i > 0 else ''
    print(f'{s:>15}  {cf[i]:>8,}    {ca:>5.1f}%   {cs:>6.1f}%  '
          f'{tf[i]:>8,}    {ta:>5.1f}%   {ts:>6.1f}%  {delta:>7}')

# ── Gráfico ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
x, w = np.arange(4), 0.32

for ax, vals_c, vals_t, title, labels in [
    (ax1, [v/cf[0]*100 for v in cf], [v/tf[0]*100 for v in tf],
     'Tasas Absolutas (desde Proyecto)', stages),
    (ax2, [cf[i]/cf[i-1]*100 for i in range(1,4)],
          [tf[i]/tf[i-1]*100 for i in range(1,4)],
     'Tasas de Paso entre Etapas',
     ['Proyecto→EL1', 'EL1→Accepted', 'Accepted→Pago'])
]:
    xi = np.arange(len(vals_c))
    bc = ax.bar(xi-w/2, vals_c, w, label='Control', color=C_CTRL, edgecolor='white', lw=1.2)
    bt = ax.bar(xi+w/2, vals_t, w, label='Test', color=C_TEST, edgecolor='white', lw=1.2)
    for bars, vals in [(bc, vals_c), (bt, vals_t)]:
        for b, v in zip(bars, vals):
            ax.text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}%',
                    ha='center', fontsize=9, fontweight='bold')
    ax.set_xticks(xi); ax.set_xticklabels(labels)
    ax.set_title(title, fontweight='bold'); ax.legend()
    ax.set_ylim(0, max(max(vals_c), max(vals_t)) * 1.2)

plt.suptitle('Análisis de Funnel: Control vs Test', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/03_funnel_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

**Hallazgo clave:** El funnel mejora en **cada etapa**, y el efecto se amplifica:

| Paso | Control | Test | Δ |
|---|---|---|---|
| Proyecto → EL1 | 62,3% | 66,2% | +3,9 pp |
| EL1 → Accepted Bid | 36,3% | 38,5% | +2,2 pp |
| **Accepted Bid → Pago** | **83,3%** | **86,4%** | **+3,1 pp** |

La mejora en Accepted→Pago es particularmente reveladora: no solo se aceptan más propuestas
en Test, sino que esas aceptaciones **se concretan en pagos con mayor frecuencia**. El nuevo
orden parece generar matches de mejor calidad — el cliente no solo acepta, realmente paga.

## 4. ¿Se contratan freelancers de mayor nivel?

El algoritmo prioriza freelancers Gold+. Si funciona, deberíamos ver un cambio
en la distribución de gamificación de los freelancers *efectivamente contratados*.

In [ ]:
# ── Gamificación de accepted bids ─────────────────────────────────────
ab_grp = accepted_bids.merge(master[['id','group']], left_on='project_id', right_on='id', how='inner')
gam_order = ['iron','bronze','silver','gold','platinum','hero']
gam_labels = ['Iron','Bronze','Silver','Gold','Platinum','Hero']
gold_plus = ['gold','platinum','hero']

gam_dist = pd.crosstab(ab_grp['worker_position_gamification'], ab_grp['group'],
                        normalize='columns').reindex(gam_order) * 100
ctrl_gp = ab_grp[ab_grp.group=='Control']['worker_position_gamification'].isin(gold_plus).mean()*100
test_gp = ab_grp[ab_grp.group=='Test']['worker_position_gamification'].isin(gold_plus).mean()*100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
x, w = np.arange(6), 0.32
cv = [gam_dist.loc[g, 'Control'] for g in gam_order]
tv = [gam_dist.loc[g, 'Test'] for g in gam_order]

ax1.bar(x-w/2, cv, w, label='Control', color=C_CTRL, edgecolor='white', lw=1.2)
ax1.bar(x+w/2, tv, w, label='Test', color=C_TEST, edgecolor='white', lw=1.2)
for xi, (c, t) in enumerate(zip(cv, tv)):
    ax1.text(xi-w/2, c+0.5, f'{c:.1f}%', ha='center', fontsize=8, fontweight='bold', color='#666')
    ax1.text(xi+w/2, t+0.5, f'{t:.1f}%', ha='center', fontsize=8, fontweight='bold', color=C_TEST)
ax1.set_xticks(x); ax1.set_xticklabels(gam_labels)
ax1.axvline(2.5, color='red', ls='--', alpha=0.3)
ax1.set_title('Distribución de Gamificación\n(Freelancers Contratados)', fontweight='bold')
ax1.set_ylabel('% de Accepted Bids'); ax1.legend()

ax2.bar(['Control','Test'], [ctrl_gp, test_gp], color=[C_CTRL, C_TEST], width=0.5, edgecolor='white', lw=1.5)
ax2.text(0, ctrl_gp+1, f'{ctrl_gp:.1f}%', ha='center', fontweight='bold', fontsize=14)
ax2.text(1, test_gp+1, f'{test_gp:.1f}%', ha='center', fontweight='bold', fontsize=14)
delta = test_gp - ctrl_gp
ax2.text(0.5, 0.91, f'Δ: {delta:+.1f} pp', transform=ax2.transAxes, ha='center',
         fontsize=12, color='#2E7D32', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', fc='#f8f7ff', ec='#ddd'))
ax2.set_title('Tasa Gold+ en Contratados\n(Gold + Platinum + Hero)', fontweight='bold')
ax2.set_ylim(0, max(ctrl_gp, test_gp) * 1.25)

plt.suptitle('Calidad del Freelancer Contratado', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/04_gamification_quality.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Gold+ → Control: {ctrl_gp:.1f}%  |  Test: {test_gp:.1f}%  |  Δ: {delta:+.1f} pp')
print(f'Bronze cayó de {cv[1]:.1f}% a {tv[1]:.1f}% (−{cv[1]-tv[1]:.1f} pp)')

**Sí, se contratan freelancers de mayor nivel.** Gold+ sube de 53,0% a 59,7% (+6,7 pp).
El cambio más drástico: Bronze cae de 26,5% a 17,3% (−9,2 pp), mientras Gold sube de 6,1% a 9,9%
y Platinum de 8,3% a 11,0%. El algoritmo cumple su función.

Punto a investigar: Iron sube levemente (9,8% → 12,4%). Podría indicar que en ciertos proyectos
no hay suficiente oferta Gold+ y el algoritmo rellena con niveles bajos.

## 5. ¿Para quién funciona mejor?

Un buen análisis no se queda en el promedio. Veamos si el efecto es uniforme
o si se concentra en algún segmento.

In [ ]:
# ── Segmentación: New vs Rebuy ────────────────────────────────────────
seg = []
for ct in ['new', 'rebuy']:
    for g in ['Control', 'Test']:
        s = master[(master.client_type==ct) & (master.group==g)]
        seg.append({'type': ct.capitalize(), 'group': g, 'n': len(s),
                    'abr': s.has_accepted_bid.mean()*100, 'el1': s.el1.mean()*100,
                    'paid': s.has_paid.mean()*100, 'prod': s.is_productive.mean()*100})
seg_df = pd.DataFrame(seg)

print('SEGMENTACIÓN POR TIPO DE CLIENTE')
print('─' * 80)
for ct in ['New', 'Rebuy']:
    c = seg_df[(seg_df.type==ct)&(seg_df.group=='Control')].iloc[0]
    t = seg_df[(seg_df.type==ct)&(seg_df.group=='Test')].iloc[0]
    lift = (t.abr/c.abr - 1) * 100
    cs = master[(master.client_type==ct.lower())&(master.group=='Control')]
    ts = master[(master.client_type==ct.lower())&(master.group=='Test')]
    _, p, *_ = z_test_prop(cs.has_accepted_bid.sum(), len(cs), ts.has_accepted_bid.sum(), len(ts))
    print(f'  {ct:5s} (n={int(c.n)+int(t.n):,})  ABR: {c.abr:.1f}% → {t.abr:.1f}%  '
          f'Lift: {lift:+.1f}%  p={p:.3f}  |  EL1: {c.el1:.1f}%→{t.el1:.1f}%  '
          f'Paid: {c.paid:.1f}%→{t.paid:.1f}%')

fig, axes = plt.subplots(1, 4, figsize=(18, 6))
for i, (col, title) in enumerate([('abr','Accepted Bid Rate'), ('el1','Tasa EL1'),
                                   ('paid','Paid Rate'), ('prod','Tasa Productiva')]):
    ax = axes[i]
    for j, ct in enumerate(['New', 'Rebuy']):
        cv = seg_df[(seg_df.type==ct)&(seg_df.group=='Control')][col].values[0]
        tv = seg_df[(seg_df.type==ct)&(seg_df.group=='Test')][col].values[0]
        ax.bar(j-0.15, cv, 0.28, color=C_CTRL, edgecolor='white', lw=1.2,
               label='Control' if j==0 else '')
        ax.bar(j+0.15, tv, 0.28, color=C_TEST, edgecolor='white', lw=1.2,
               label='Test' if j==0 else '')
        ax.text(j-0.15, cv+0.4, f'{cv:.1f}%', ha='center', fontsize=8, fontweight='bold')
        ax.text(j+0.15, tv+0.4, f'{tv:.1f}%', ha='center', fontsize=8, fontweight='bold')
    ax.set_xticks([0,1]); ax.set_xticklabels(['New','Rebuy'])
    ax.set_title(title, fontweight='bold', fontsize=11)
    if i == 0: ax.legend(fontsize=9)

plt.suptitle('Métricas por Tipo de Cliente: New vs Rebuy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/05_segmentation_client_type.png', dpi=150, bbox_inches='tight')
plt.show()

**Hallazgo más importante del análisis:** El efecto en clientes **New es 5× mayor** que en Rebuy.

| Segmento | Control ABR | Test ABR | Lift |
|---|---|---|---|
| **New** (n=1.493) | 17,2% | 20,2% | **+17,7%** |
| Rebuy (n=741) | 34,3% | 35,5% | +3,5% |

Tiene lógica causal directa: un cliente nuevo no conoce la plataforma, no tiene freelancers
favoritos, y depende completamente del orden de presentación. Un cliente recurrente ya sabe
qué buscar. **El nuevo orden ayuda más a quienes más lo necesitan.**

In [ ]:
# ── Segmentación por país (Top 5) ─────────────────────────────────────
top5 = master.user_country.value_counts().head(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
x, w, cr, tr = np.arange(5), 0.32, [], []

print('SEGMENTACIÓN POR PAÍS (Top 5) — Accepted Bid Rate')
print('─' * 85)
for country in top5:
    cs = master[(master.user_country==country)&(master.group=='Control')]
    ts = master[(master.user_country==country)&(master.group=='Test')]
    c_r, t_r = cs.has_accepted_bid.mean()*100, ts.has_accepted_bid.mean()*100
    cr.append(c_r); tr.append(t_r)
    _, p, *_ = z_test_prop(cs.has_accepted_bid.sum(), len(cs), ts.has_accepted_bid.sum(), len(ts))
    print(f'  {country}  Control: {c_r:5.1f}% (n={len(cs)})  Test: {t_r:5.1f}% (n={len(ts)})  '
          f'Δ: {t_r-c_r:+.1f}pp  p={p:.3f}')

ax.bar(x-w/2, cr, w, label='Control', color=C_CTRL, edgecolor='white', lw=1.2)
ax.bar(x+w/2, tr, w, label='Test', color=C_TEST, edgecolor='white', lw=1.2)
for xi, (c, t) in enumerate(zip(cr, tr)):
    ax.text(xi-w/2, c+0.3, f'{c:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#555')
    ax.text(xi+w/2, t+0.3, f'{t:.1f}%', ha='center', fontsize=9, fontweight='bold', color=C_TEST)
    d = t - c
    ax.text(xi, max(c,t)+2.5, f'{d:+.1f}pp', ha='center', fontsize=9, fontweight='bold',
            color='#2E7D32' if d > 0 else '#C62828')
ax.set_xticks(x)
ax.set_xticklabels([f'{c}\n(n={master[master.user_country==c].shape[0]})' for c in top5])
ax.set_ylabel('Accepted Bid Rate (%)'); ax.legend(fontsize=11)
ax.set_title('Accepted Bid Rate por País (Top 5)', fontweight='bold', fontsize=13)
ax.set_ylim(0, max(max(cr),max(tr))*1.35)
plt.tight_layout()
plt.savefig('output/06_segmentation_country.png', dpi=150, bbox_inches='tight')
plt.show()

Argentina muestra la señal más fuerte (+8,8 pp, lift +52,5%) pero con N insuficiente (n=125).
Brasil, el mercado principal, tiene efecto moderado (+2,1 pp). México es el único país con efecto
negativo (−3,1 pp), algo a investigar, aunque no significativo (p = 0,65).

## 6. ¿El efecto es estable en el tiempo?

In [ ]:
# ── Evolución semanal ─────────────────────────────────────────────────
wk = []
for week in sorted(master.week.unique()):
    for g in ['Control', 'Test']:
        s = master[(master.week==week)&(master.group==g)]
        if len(s) == 0: continue
        wk.append({'week': week, 'lbl': f'Sem {week}', 'group': g, 'n': len(s),
                   'abr': s.has_accepted_bid.mean()*100,
                   'el1': s.el1.mean()*100, 'paid': s.has_paid.mean()*100})
wk_df = pd.DataFrame(wk)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, (col, title) in enumerate([('abr','Accepted Bid Rate (%)'), ('el1','Tasa EL1 (%)'),
                                   ('paid','Paid Rate (%)')]):
    ax = axes[i]
    for g, color, mk in [('Control',C_CTRL,'o'), ('Test',C_TEST,'s')]:
        d = wk_df[wk_df.group==g].sort_values('week')
        ax.plot(d.lbl, d[col], marker=mk, lw=2.5, ms=8, label=g, color=color)
        for _, r in d.iterrows():
            off = 1.2 if g == 'Test' else -1.5
            ax.text(r.lbl, r[col]+off, f'{r[col]:.1f}%', ha='center', fontsize=8,
                    fontweight='bold', color=color)
    ax.set_title(title, fontweight='bold', fontsize=12); ax.legend(fontsize=10)

plt.suptitle('Evolución Temporal Semanal: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/07_weekly_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

for week in sorted(wk_df.week.unique()):
    c = wk_df[(wk_df.week==week)&(wk_df.group=='Control')].iloc[0]
    t = wk_df[(wk_df.week==week)&(wk_df.group=='Test')].iloc[0]
    print(f'  Sem {week}  Control: {c.abr:5.1f}% (n={int(c.n)})  '
          f'Test: {t.abr:5.1f}% (n={int(t.n)})  Δ: {t.abr-c.abr:+.1f}pp')

**Precaución:** El efecto no es estable semana a semana. Semana 28 concentra el mayor delta
(+6,5 pp) mientras Semana 29 es prácticamente nula (+0,1 pp). Con ~250 proyectos/semana/grupo
la varianza semanal es alta, pero refuerza que necesitamos **más tiempo de observación**.

## 7. ¿Por qué no alcanzamos significancia?

Antes de concluir, respondamos una pregunta crítica: ¿el test tenía suficiente
poder para detectar el efecto que observamos?

In [ ]:
# ── Power analysis ────────────────────────────────────────────────────
def n_needed(p_c, p_t, alpha=0.05, power=0.80):
    za, zb = norm.ppf(1-alpha/2), norm.ppf(power)
    pa = (p_c + p_t) / 2
    return int(np.ceil(((za*np.sqrt(2*pa*(1-pa)) + zb*np.sqrt(p_c*(1-p_c)+p_t*(1-p_t))) / (p_t-p_c))**2))

def current_power(nc, nt, pc, pt, alpha=0.05):
    pp = (pc*nc + pt*nt)/(nc+nt)
    se = np.sqrt(pp*(1-pp)*(1/nc+1/nt))
    return 1 - norm.cdf(norm.ppf(1-alpha/2) - abs(pt-pc)/se)

pc = ctrl.has_accepted_bid.mean()
pt = test.has_accepted_bid.mean()
pwr = current_power(len(ctrl), len(test), pc, pt)
n80 = n_needed(pc, pt, power=0.80)
n90 = n_needed(pc, pt, power=0.90)
rate = len(master) / 18

print('POWER ANALYSIS — Accepted Bid Rate')
print('─' * 55)
print(f'  Efecto observado:  {pc:.1%} → {pt:.1%}  (Δ = {(pt-pc)*100:+.2f} pp)')
print(f'  Poder actual:      {pwr:.1%}  ← INSUFICIENTE (se requiere ≥80%)')
print(f'  N actual/grupo:    ~{len(ctrl):,}')
print(f'  N para 80% poder:  {n80:,}  (+{max(0,(n80*2-len(master))/rate):.0f} días)')
print(f'  N para 90% poder:  {n90:,}  (+{max(0,(n90*2-len(master))/rate):.0f} días)')

**Ahí está la respuesta.** Con ~1.100 proyectos por grupo, el test tenía solo **35% de poder** —
es decir, *aunque el efecto sea real*, teníamos solo un 35% de probabilidad de detectarlo.
Es como buscar algo en la oscuridad con una linterna demasiado débil: que no lo veamos no
significa que no esté ahí.

Para 80% de poder necesitamos ~3.600 proyectos/grupo → **~40 días adicionales** al ritmo actual.

---

## 8. Conclusión: ¿Qué hacemos con esto?

### La evidencia a favor

1. **Consistencia direccional total.** Las 4 métricas de conversión apuntan en la misma dirección. Si bien están correlacionadas por ser parte del mismo funnel, que *todas* mejoren simultáneamente es una señal fuerte de efecto real, no de variación aleatoria.
2. **Amplificación en el funnel.** El lift crece de EL1 (+6%) a Paid Rate (+17%) — patrón coherente, no ruido.
3. **Mejora en calidad.** Gold+ sube de 53% a 60%. Bronze cae de 27% a 17%.
4. **P-values borderline.** EL1 (p = 0,056) y Paid Rate (p = 0,066) están muy cerca del umbral.
5. **Guardrail limpio.** Bids promedio estable (p = 0,59).

### La evidencia que pide cautela

1. Ninguna métrica de conversión es significativa a α = 0,05 (ni con corrección por comparaciones múltiples).
2. Evolución semanal irregular (Sem 28: +6,5 pp vs Sem 29: +0,1 pp).
3. México con efecto negativo (−3,1 pp, p = 0,65).
4. **Test underpowered** (35% de poder).

### Decisión: **ITERAR — Extender el test antes de escalar**

**No descartar:** la señal es demasiado consistente (4 métricas, funnel completo, calidad Gold+).

**No escalar hoy:** sin significancia, el IC 95% del ABR incluye cero (−0,72 pp a +6,37 pp).

**Extender ~6 semanas más** → ~3.600 proyectos/grupo → 80% de poder.

| Escenario | Costo |
|---|---|
| **Extender** | Bajo — mantener el 50/50 split no impacta el negocio |
| **Escalar incorrectamente** | Alto — si es ruido, degradamos la experiencia |
| **Descartar incorrectamente** | El más alto — +3 pp en ABR = ~60 conversiones extra por cada 2.000 proyectos |

---

## 9. Próximos pasos

| # | Acción | Horizonte | Fundamento (basado en los datos) |
|---|---|---|---|
| **1** | **Extender el test ~6 semanas** | Inmediato | Poder = 35%. Necesitamos 3.586/grupo para 80%. Pre-registrar tamaño y métrica para evitar sesgo de *peeking*. |
| **2** | **Desagregar por categoría** | En paralelo | Heterogeneidad por país (AR +52% vs MX −15%) sugiere heterogeneidad por vertical. Escalar parcialmente si 3-4 categorías son significativas. |
| **3** | **Focalizar en clientes New** | 2ª iteración | Lift 5× mayor (+17,7% vs +3,5%). Un test solo para New lograría significancia más rápido. |
| **4** | **Investigar Iron en Test** | En paralelo | Sube 9,8%→12,4% pese al algoritmo. ¿Falta de oferta Gold+? ¿Efecto de contraste? |
| **5** | **Medir GMV total** | En extensión | Ticket baja −6% pero GMV total sube +5,4% (~$33.200 vs ~$31.500) por mayor volumen. |
| **6** | **Iterar pesos del algoritmo** | Post-confirmación | Skills matching (10%) podría subir. Gamificación (30%) podría estar sobredimensionada. |
| **7** | **Expandir categorías** | Post-confirmación | Rollout escalonado a verticales fuera del scope actual. |

---

*Herramientas: Python 3.13 · pandas · SciPy (stats) · Matplotlib · Seaborn*
*Tests estadísticos: z-test de dos proporciones · t-test de Welch · Análisis de potencia*